In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns 

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer , KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

In [2]:
# reading file
hotel_data = pd.read_csv(r'../data/raw/hotel_bookings.csv')
hotel_data.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [3]:
%load_ext autoreload
%autoreload 2

import sys

# 1. Add the main project root folder to sys.path (NOT the file itself)
sys.path.append('..')

from src.hotel_booking_cancelation_prediction.cleaning import pandas_cleaning
df = pandas_cleaning(hotel_data)

if 'company' in df.columns :
        df['has_company'] = df['company'].notnull().astype('int64')
        df = df.drop(columns = ['company'])

    
df.info()

<class 'pandas.DataFrame'>
Index: 87229 entries, 0 to 119389
Data columns (total 31 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   hotel                           87229 non-null  str    
 1   is_canceled                     87229 non-null  int64  
 2   lead_time                       87229 non-null  int64  
 3   arrival_date_year               87229 non-null  int64  
 4   arrival_date_month              87229 non-null  str    
 5   arrival_date_week_number        87229 non-null  int64  
 6   arrival_date_day_of_month       87229 non-null  int64  
 7   stays_in_weekend_nights         87229 non-null  int64  
 8   stays_in_week_nights            87229 non-null  int64  
 9   adults                          87229 non-null  int64  
 10  children                        87225 non-null  float64
 11  babies                          87229 non-null  int64  
 12  meal                            87229 non-null 

In [4]:
x = df.drop(columns = ['is_canceled'])
y = df['is_canceled']

In [5]:
# spiliting x and y into train and test
x_train , x_test , y_train , y_test = train_test_split(x , y , test_size = 0.3 , random_state = 0)

In [6]:
print('x_train shape :' , x_train.shape)
print('x_test shape :' , x_test.shape)

x_train shape : (61060, 30)
x_test shape : (26169, 30)


## Imputation techniques

In [7]:
# getting numerical and categorical cols 

num_cols = x_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = x_train.select_dtypes(include=['object', 'str']).columns.tolist()

# Ensure leak/date columns are excluded from categorical columns
for col in ['reservation_status_date', 'reservation_status']:
    if col in cat_cols:
        cat_cols.remove(col)


In [8]:
# 1 mean column transformer
passthrough_cols = [c for c in x_train.columns if c not in num_cols + cat_cols] # those cols which are present in x_train like company but 
# not in num_cols or cat_cols that we are passing into our ct_mean

ct_mean = ColumnTransformer(transformers = [
    ('num_mean' , SimpleImputer(strategy = 'mean') , num_cols),
    ('cat_most_frequent' , SimpleImputer(strategy = 'most_frequent') , cat_cols)
],remainder = 'passthrough'
)
X_train_mean = pd.DataFrame(ct_mean.fit_transform(x_train), columns=num_cols + cat_cols + passthrough_cols, index=x_train.index)
X_test_mean = pd.DataFrame(ct_mean.transform(x_test), columns=num_cols + cat_cols + passthrough_cols, index=x_test.index)

In [9]:
# 2 median for num and mode for cat
ct_median = ColumnTransformer(transformers = [
    ('num_median' , SimpleImputer(strategy = 'median') , num_cols),
    ('cat_mode' , SimpleImputer(strategy = 'most_frequent'),cat_cols)
],remainder = 'passthrough'
)
X_train_median = pd.DataFrame(ct_median.fit_transform(x_train) , columns = num_cols + cat_cols + passthrough_cols , index = x_train.index)
X_test_median = pd.DataFrame(ct_median.transform(x_test) , columns = num_cols + cat_cols + passthrough_cols , index = x_test.index)

In [10]:
# 3 constant value on cols
ct_const = ColumnTransformer(transformers=[
    ('num', SimpleImputer(strategy='constant', fill_value=0), num_cols),
    ('cat', SimpleImputer(strategy='constant', fill_value='Unknown'), cat_cols)
], remainder='passthrough')

X_train_const = pd.DataFrame(ct_const.fit_transform(x_train), columns=num_cols + cat_cols + passthrough_cols, index=x_train.index)
X_test_const = pd.DataFrame(ct_const.transform(x_test), columns=num_cols + cat_cols + passthrough_cols, index=x_test.index)

In [11]:
# 4 KNN Imputer (Numerical) + Constant 'Unknown' (Categorical)
ct_knn = ColumnTransformer(transformers=[
    ('num', KNNImputer(n_neighbors=5), num_cols),
    ('cat', SimpleImputer(strategy='constant', fill_value='Unknown'), cat_cols)
], remainder='passthrough')

X_train_knn = pd.DataFrame(ct_knn.fit_transform(x_train), columns=num_cols + cat_cols + passthrough_cols, index=x_train.index)
X_test_knn = pd.DataFrame(ct_knn.transform(x_test), columns=num_cols + cat_cols + passthrough_cols, index=x_test.index)

In [12]:
# 5 MICE / Iterative Imputer (Numerical) + Constant 'Unknown' (Categorical)
ct_mice = ColumnTransformer(transformers=[
    ('num', IterativeImputer(max_iter=10, random_state=42), num_cols),
    ('cat', SimpleImputer(strategy='constant', fill_value='Unknown'), cat_cols)
], remainder='passthrough')
X_train_mice = pd.DataFrame(ct_mice.fit_transform(x_train), columns=num_cols + cat_cols + passthrough_cols, index=x_train.index)
X_test_mice = pd.DataFrame(ct_mice.transform(x_test), columns=num_cols + cat_cols + passthrough_cols, index=x_test.index)

In [13]:
import optuna
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import cross_val_score

def objective(trial):
    # 1. Choose Numerical Imputer
    num_imputer_name = trial.suggest_categorical(
        'num_imputer', ['mean', 'median', 'constant', 'knn', 'iterative']
    )

    if num_imputer_name == 'mean':
        num_imputer = SimpleImputer(strategy='mean')
    elif num_imputer_name == 'median':
        num_imputer = SimpleImputer(strategy='median')
    elif num_imputer_name == 'constant':
        num_imputer = SimpleImputer(strategy='constant', fill_value=0)
    elif num_imputer_name == 'knn':
        n_neighbors = trial.suggest_int('knn_n_neighbors', 3, 7)
        num_imputer = KNNImputer(n_neighbors=n_neighbors)
    elif num_imputer_name == 'iterative':
        max_iter = trial.suggest_int('iterative_max_iter', 5, 10)
        num_imputer = IterativeImputer(max_iter=max_iter, random_state=42)

    # 2. Choose Categorical Imputer
    cat_imputer_name = trial.suggest_categorical(
        'cat_imputer', ['most_frequent', 'constant']
    )
    
    if cat_imputer_name == 'most_frequent':
        cat_imputer = SimpleImputer(strategy='most_frequent')
    else:
        cat_imputer = SimpleImputer(strategy='constant', fill_value='Unknown')

    # 3. Categorical Pipeline: Impute -> One-Hot Encode
    cat_pipeline = Pipeline(steps=[
        ('imputer', cat_imputer),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])

    # 4. Combined Preprocessor
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', num_imputer, num_cols),
            ('cat', cat_pipeline, cat_cols)
        ],
        remainder='drop'  # Automatically drops date/leakage columns
    )

    # 5. Full Pipeline
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(
            n_estimators=50, max_depth=10, random_state=42, n_jobs=-1
        ))
    ])

    # 6. Sample 10,000 rows to keep cross-validation fast
    sample_x = x_train.sample(n=min(10000, len(x_train)), random_state=42)
    sample_y = y_train.loc[sample_x.index]

    # 7. Evaluate Cross-Validation Score
    score = cross_val_score(
        pipeline, sample_x, sample_y,
        cv=3, scoring='roc_auc', n_jobs=-1
    ).mean()

    return score

# Run Optuna Study
sampler = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(direction='maximize',sampler = sampler)
study.optimize(objective, n_trials=15)

print("=" * 60)
print(f"WINNING IMPUTER COMBINATION: {study.best_params}")
print(f"BEST CV ROC-AUC SCORE: {study.best_value:.4f}")
print("=" * 60)

c:\Users\lenovo\Desktop\HOTEL_BOOKING_CANCELATION_PREDICTION\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-08-02 23:20:43,026] A new study created in memory with name: no-name-5c7756bb-231a-49c6-8a2a-c70b9326f0a9
[I 2026-08-02 23:20:55,175] Trial 0 finished with value: 0.8596257323093441 and parameters: {'num_imputer': 'median', 'cat_imputer': 'most_frequent'}. Best is trial 0 with value: 0.8596257323093441.
[I 2026-08-02 23:21:03,373] Trial 1 finished with value: 0.8573710548199381 and parameters: {'num_imputer': 'iterative', 'iterative_max_iter': 9, 'cat_imputer': 'most_frequent'}. Best is trial 0 with value: 0.8596257323093441.
[I 2026-08-02 23:21:09,747] Trial 2 finished with value: 0.8604746873160972 and parameters: {'num_imputer': 'constant', 'cat_imputer': 'most_frequent'}. Best is trial 2 w

WINNING IMPUTER COMBINATION: {'num_imputer': 'constant', 'cat_imputer': 'constant'}
BEST CV ROC-AUC SCORE: 0.8623


In [14]:
# by optuna we should use constant to impute on both categorical and numerical cols 
# therefore what we did in pandas is correct therefore we will use 
# processed data further on


In [15]:
# by optuna we should use constant strategy
final_preprocessor = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='constant', fill_value=0), num_cols),
        ('cat', SimpleImputer(strategy='constant', fill_value='Unknown'), cat_cols)
    ],
    remainder='drop'  # Automatically drops date and target leakage columns
)

# 2. Fit on x_train, transform both train and test
x_train_imputed = pd.DataFrame(
    final_preprocessor.fit_transform(x_train),
    columns=num_cols + cat_cols,
    index=x_train.index
)

x_test_imputed = pd.DataFrame(
    final_preprocessor.transform(x_test),
    columns=num_cols + cat_cols,
    index=x_test.index
)

# 3. Enforce original numerical data types (ColumnTransformer outputs object/float dtypes)
for col in num_cols:
    x_train_imputed[col] = pd.to_numeric(x_train_imputed[col])
    x_test_imputed[col] = pd.to_numeric(x_test_imputed[col])

if 'children' in x_train_imputed.columns:
    x_train_imputed['children'] = x_train_imputed['children'].astype('int64')
    x_test_imputed['children'] = x_test_imputed['children'].astype('int64')

print("x_train_imputed shape:", x_train_imputed.shape)
print("x_test_imputed shape:", x_test_imputed.shape)
print("Remaining NaNs in x_train_imputed:", x_train_imputed.isnull().sum().sum())
print("Remaining NaNs in x_test_imputed:", x_test_imputed.isnull().sum().sum())

x_train_imputed shape: (61060, 29)
x_test_imputed shape: (26169, 29)
Remaining NaNs in x_train_imputed: 0
Remaining NaNs in x_test_imputed: 0


In [16]:
x_train_imputed.info()

<class 'pandas.DataFrame'>
Index: 61060 entries, 2155 to 96522
Data columns (total 29 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   lead_time                       61060 non-null  float64
 1   arrival_date_year               61060 non-null  float64
 2   arrival_date_week_number        61060 non-null  float64
 3   arrival_date_day_of_month       61060 non-null  float64
 4   stays_in_weekend_nights         61060 non-null  float64
 5   stays_in_week_nights            61060 non-null  float64
 6   adults                          61060 non-null  float64
 7   children                        61060 non-null  int64  
 8   babies                          61060 non-null  float64
 9   is_repeated_guest               61060 non-null  float64
 10  previous_cancellations          61060 non-null  float64
 11  previous_bookings_not_canceled  61060 non-null  float64
 12  booking_changes                 61060 non-nul

In [17]:
# List of columns that logically should be integers
int_cols = [
    'lead_time', 'arrival_date_year', 'arrival_date_week_number', 
    'arrival_date_day_of_month', 'stays_in_weekend_nights', 
    'stays_in_week_nights', 'adults', 'children', 'babies', 
    'is_repeated_guest', 'previous_cancellations', 
    'previous_bookings_not_canceled', 'booking_changes', 
    'days_in_waiting_list', 'required_car_parking_spaces', 
    'total_of_special_requests', 'has_company'
]

# Cast integer columns back to int64
for col in int_cols:
    if col in x_train_imputed.columns:
        x_train_imputed[col] = x_train_imputed[col].astype('int64')
        x_test_imputed[col] = x_test_imputed[col].astype('int64')

# Verify the types
x_train_imputed.dtypes

lead_time                           int64
arrival_date_year                   int64
arrival_date_week_number            int64
arrival_date_day_of_month           int64
stays_in_weekend_nights             int64
stays_in_week_nights                int64
adults                              int64
children                            int64
babies                              int64
is_repeated_guest                   int64
previous_cancellations              int64
previous_bookings_not_canceled      int64
booking_changes                     int64
agent                             float64
days_in_waiting_list                int64
adr                               float64
required_car_parking_spaces         int64
total_of_special_requests           int64
has_company                         int64
hotel                                 str
arrival_date_month                    str
meal                                  str
country                               str
market_segment                    

## Outlier handling

In [18]:
# Selected features for outlier detection
# excluding categorical variables and binary flags like has_company or is_repeated_guest.
outlier_cols = [
    'lead_time', 
    'stays_in_weekend_nights', 
    'stays_in_week_nights', 
    'adults', 
    'children', 
    'babies', 
    'booking_changes', 
    'days_in_waiting_list', 
    'adr'
]

In [19]:


outlier_summary = []

for col in outlier_cols:
    # 1. Percentile Bounds (1st and 99th)
    p_lower = x_train_imputed[col].quantile(0.01)
    p_upper = x_train_imputed[col].quantile(0.99)
    p_count = ((x_train_imputed[col] < p_lower) | (x_train_imputed[col] > p_upper)).sum()
    
    # 2. IQR Bounds (1.5 * IQR)
    q1 = x_train_imputed[col].quantile(0.25)
    q3 = x_train_imputed[col].quantile(0.75)
    iqr = q3 - q1
    iqr_lower = q1 - 1.5 * iqr
    iqr_upper = q3 + 1.5 * iqr
    iqr_count = ((x_train_imputed[col] < iqr_lower) | (x_train_imputed[col] > iqr_upper)).sum()
    
    # 3. Z-Score Bounds (|Z| > 3)
    # Using ddof=0 for standard z-score
    z_scores = np.abs((x_train_imputed[col] - x_train_imputed[col].mean()) / x_train_imputed[col].std())
    z_count = (z_scores > 3).sum()
    
    outlier_summary.append({
        'Feature': col,
        'Percentile Outliers (<1% or >99%)': p_count,
        'IQR Outliers (<Q1-1.5IQR or >Q3+1.5IQR)': iqr_count,
        'Z-Score Outliers (|Z| > 3)': z_count
    })

# Convert summary into a readable DataFrame
outlier_df = pd.DataFrame(outlier_summary)
outlier_df

,Feature,Percentile Outliers (<1% or >99%),IQR Outliers (<Q1-1.5IQR or >Q3+1.5IQR),Z-Score Outliers (|Z| > 3)
0,lead_time,607,1663,749
1,stays_in_weekend_nights,194,150,194
2,stays_in_week_nights,236,1038,1038
3,adults,212,15930,212
4,children,52,5896,2628
5,babies,16,657,657
6,booking_changes,412,11056,1016
7,days_in_waiting_list,600,600,442
8,adr,597,1768,592


In [20]:

from sklearn.preprocessing import OneHotEncoder

# Helper function to dynamically cap outliers directly on DataFrames
def cap_outliers_df(df, cols, method, threshold):
    df_capped = df.copy()
    
    if method == 'none':
        return df_capped

    for col in cols:
        if method == 'percentile':
            lower = df_capped[col].quantile(threshold)
            upper = df_capped[col].quantile(1.0 - threshold)
        elif method == 'iqr':
            q1 = df_capped[col].quantile(0.25)
            q3 = df_capped[col].quantile(0.75)
            iqr = q3 - q1
            lower = q1 - (threshold * iqr)
            upper = q3 + (threshold * iqr)
        elif method == 'zscore':
            mean = df_capped[col].mean()
            std = df_capped[col].std()
            lower = mean - (threshold * std)
            upper = mean + (threshold * std)

        df_capped[col] = df_capped[col].clip(lower=lower, upper=upper)
        
    return df_capped


def objective(trial):
    
    # 1. Choose Outlier Strategy & Threshold
    outlier_method = trial.suggest_categorical(
        'outlier_method', ['percentile', 'iqr', 'zscore']
    )

    if outlier_method == 'percentile':
        outlier_threshold = trial.suggest_float('percentile_cutoff', 0.005, 0.05)
    elif outlier_method == 'iqr':
        outlier_threshold = trial.suggest_float('iqr_multiplier', 1.5, 3.0)
    elif outlier_method == 'zscore':
        outlier_threshold = trial.suggest_float('zscore_threshold', 2.5, 4.0)
    else:
        outlier_threshold = None

    
    cat_pipeline = Pipeline(steps=[
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])

    # 5. Combined Preprocessor
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', 'passthrough', num_cols),
            ('cat', cat_pipeline, cat_cols)
        ],
        remainder='drop'  # Automatically drops date/leakage columns
    )

    # 6. Sample 10,000 rows from YOUR existing x_train to keep Optuna fast
    sample_x = x_train_imputed.sample(n=min(10000, len(x_train)), random_state=42)
    sample_y = y_train.loc[sample_x.index]

    # Apply outlier capping to the sampled training data
    sample_x_capped = cap_outliers_df(sample_x, outlier_cols, outlier_method, outlier_threshold)

    # 7. Build Full Pipeline
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(
            n_estimators=50, max_depth=10, random_state=42, n_jobs=-1
        ))
    ])

    # 8. Evaluate Cross-Validation Score using cross_val_score
    score = cross_val_score(
        pipeline, sample_x_capped, sample_y,
        cv=3, scoring='roc_auc', n_jobs=-1
    ).mean()

    return score

# Run Optuna Study
study = optuna.create_study(direction='maximize',sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=15)

print("=" * 60)
print(f"WINNING COMBINATION: {study.best_params}")
print(f"BEST CV ROC-AUC SCORE: {study.best_value:.4f}")
print("=" * 60)

[I 2026-08-02 23:21:34,676] A new study created in memory with name: no-name-20178a9e-ceda-45ca-8d7b-1727479f0dd4
[I 2026-08-02 23:21:36,103] Trial 0 finished with value: 0.8608025146179396 and parameters: {'outlier_method': 'iqr', 'iqr_multiplier': 2.397987726295555}. Best is trial 0 with value: 0.8608025146179396.
[I 2026-08-02 23:21:37,474] Trial 1 finished with value: 0.8600714738005903 and parameters: {'outlier_method': 'percentile', 'percentile_cutoff': 0.04397792655987209}. Best is trial 0 with value: 0.8608025146179396.
[I 2026-08-02 23:21:38,911] Trial 2 finished with value: 0.8605822452116888 and parameters: {'outlier_method': 'iqr', 'iqr_multiplier': 2.9548647782429915}. Best is trial 0 with value: 0.8608025146179396.
[I 2026-08-02 23:21:40,350] Trial 3 finished with value: 0.8604033968710799 and parameters: {'outlier_method': 'percentile', 'percentile_cutoff': 0.013253202943404523}. Best is trial 0 with value: 0.8608025146179396.
[I 2026-08-02 23:21:42,316] Trial 4 finished

WINNING COMBINATION: {'outlier_method': 'zscore', 'zscore_threshold': 3.8484811634335334}
BEST CV ROC-AUC SCORE: 0.8635


In [21]:
# Apply the winning outlier capping rule from Optuna
z_thresh = study.best_params['zscore_threshold']

x_train_capped = x_train_imputed.copy()
x_test_capped = x_test_imputed.copy()

for col in outlier_cols:
    # Compute mean and std strictly on x_train (prevents leakage)
    mean_val = x_train_imputed[col].mean()
    std_val = x_train_imputed[col].std()
    
    lower_bound = mean_val - (z_thresh * std_val)
    upper_bound = mean_val + (z_thresh * std_val)
    
    # Clip values within Z-score bounds
    x_train_capped[col] = x_train_capped[col].clip(lower=lower_bound, upper=upper_bound)
    x_test_capped[col] = x_test_capped[col].clip(lower=lower_bound, upper=upper_bound)

print(f"✅ Applied Z-Score Outlier Capping (|Z| > {z_thresh:.4f}) to numeric columns!")

✅ Applied Z-Score Outlier Capping (|Z| > 3.8485) to numeric columns!


In [22]:
print("--- FINAL VERIFICATION ---")
print(f"Original Imputed Max ADR : {x_train_imputed['adr'].max()}")
print(f"Capped Max ADR          : {x_train_capped['adr'].max()}")
print(f"Missing values in Train : {x_train_capped.isnull().sum().sum()}")
print(f"Missing values in Test  : {x_test_capped.isnull().sum().sum()}")

--- FINAL VERIFICATION ---
Original Imputed Max ADR : 510.0
Capped Max ADR          : 306.89953026498586
Missing values in Train : 0
Missing values in Test  : 0


## Feature construction

In [23]:
# 1. Create clean working copies for feature engineering
x_train_fe = x_train_capped.copy()
x_test_fe = x_test_capped.copy()

# Apply transformations to both splits
for df_fe in [x_train_fe, x_test_fe]:
    # A. Total Stay Duration
    df_fe['total_stay_nights'] = df_fe['stays_in_weekend_nights'] + df_fe['stays_in_week_nights']
    df_fe['is_weekend_only'] = ((df_fe['stays_in_weekend_nights'] > 0) & 
                                (df_fe['stays_in_week_nights'] == 0)).astype(int)

    # B. Headcount & Composition Flags
    df_fe['total_guests'] = df_fe['adults'] + df_fe['children'] + df_fe['babies']
    df_fe['is_family'] = ((df_fe['children'] > 0) | (df_fe['babies'] > 0)).astype(int)
    df_fe['is_solo'] = ((df_fe['adults'] == 1) & (df_fe['total_guests'] == 1)).astype(int)

    # C. Financial Ratios (avoiding division by zero)
    df_fe['adr_per_person'] = df_fe['adr'] / np.maximum(df_fe['total_guests'], 1)

    # D. Room Swap Flag (Strong cancellation predictor)
    df_fe['room_type_changed'] = (df_fe['reserved_room_type'] != df_fe['assigned_room_type']).astype(int)

print("✅ Feature Engineering Complete!")
print(f"New Features Added: {x_train_fe.shape[1] - x_train_capped.shape[1]}")
print(f"x_train_fe shape : {x_train_fe.shape}")
print(f"x_test_fe shape  : {x_test_fe.shape}")

✅ Feature Engineering Complete!
New Features Added: 7
x_train_fe shape : (61060, 36)
x_test_fe shape  : (26169, 36)


## Encoding categorical cols

In [24]:
cardinality = x_train_fe[cat_cols].nunique().sort_values(ascending=False)
print(cardinality)

country                 167
arrival_date_month       12
assigned_room_type       11
reserved_room_type        9
market_segment            8
distribution_channel      5
customer_type             4
meal                      4
deposit_type              3
hotel                     2
dtype: int64


In [25]:
from sklearn.preprocessing import OneHotEncoder

# Threshold: columns with more unique values than this get frequency-encoded instead of one-hot
CARDINALITY_THRESHOLD = 15

cardinality = x_train_fe[cat_cols].nunique()
low_card_cols = cardinality[cardinality <= CARDINALITY_THRESHOLD].index.tolist()
high_card_cols = cardinality[cardinality > CARDINALITY_THRESHOLD].index.tolist()

print("One-hot encoding on these (low cardinality):", low_card_cols)
print("Frequency encoding on these (high cardinality):", high_card_cols)

# ---- ONE-HOT ENCODING for low-cardinality columns ----
x_train_ohe = pd.DataFrame(index=x_train_fe.index)
x_test_ohe = pd.DataFrame(index=x_test_fe.index)

if low_card_cols:
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    ohe.fit(x_train_fe[low_card_cols])

    train_ohe_array = ohe.transform(x_train_fe[low_card_cols])
    test_ohe_array = ohe.transform(x_test_fe[low_card_cols])

    ohe_col_names = ohe.get_feature_names_out(low_card_cols)

    x_train_ohe = pd.DataFrame(train_ohe_array, columns=ohe_col_names, index=x_train_fe.index)
    x_test_ohe = pd.DataFrame(test_ohe_array, columns=ohe_col_names, index=x_test_fe.index)

# ---- FREQUENCY ENCODING for high-cardinality columns ----
x_train_freq = pd.DataFrame(index=x_train_fe.index)
x_test_freq = pd.DataFrame(index=x_test_fe.index)

for col in high_card_cols:
    # Compute frequency map from TRAIN only (prevents leakage)
    freq_map = x_train_fe[col].value_counts(normalize=True)

    x_train_freq[col + '_freq'] = x_train_fe[col].map(freq_map)
    # Unseen categories in test get frequency 0
    x_test_freq[col + '_freq'] = x_test_fe[col].map(freq_map).fillna(0)

engineered_num_cols = [
    'total_stay_nights', 'is_weekend_only', 'total_guests',
    'is_family', 'is_solo', 'adr_per_person', 'room_type_changed'
]
all_num_cols = num_cols + engineered_num_cols

# ---- COMBINE everything: numeric + one-hot + frequency ----
x_train_final = pd.concat([x_train_fe[all_num_cols], x_train_ohe, x_train_freq], axis=1)
x_test_final = pd.concat([x_test_fe[all_num_cols], x_test_ohe, x_test_freq], axis=1)

print("\nx_train_final shape:", x_train_final.shape)
print("x_test_final shape:", x_test_final.shape)
print("Any NaNs in x_train_final:", x_train_final.isnull().sum().sum())
print("Any NaNs in x_test_final:", x_test_final.isnull().sum().sum())

One-hot encoding on these (low cardinality): ['hotel', 'arrival_date_month', 'meal', 'market_segment', 'distribution_channel', 'reserved_room_type', 'assigned_room_type', 'deposit_type', 'customer_type']
Frequency encoding on these (high cardinality): ['country']

x_train_final shape: (61060, 85)
x_test_final shape: (26169, 85)
Any NaNs in x_train_final: 0
Any NaNs in x_test_final: 0


## Transformation and scaling

In [26]:
import optuna
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import PowerTransformer, StandardScaler, MinMaxScaler, RobustScaler
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

continuous_cols = num_cols + ['country_freq', 'total_stay_nights', 'total_guests', 'adr_per_person']
continuous_cols = [c for c in continuous_cols if c in x_train_final.columns]

def objective(trial):
    # 1. Choose transformation method
    transform_method = trial.suggest_categorical(
        'transform_method', ['none', 'yeo-johnson','log1p']
    )

    # 2. Choose scaler
    scaler_method = trial.suggest_categorical(
        'scaler_method', ['none', 'standard', 'minmax', 'robust']
    )

    # Build transformer step
    if transform_method == 'yeo-johnson':
        transformer_step = PowerTransformer(method='yeo-johnson', standardize=False)
    elif transform_method == 'log1p':
        transformer_step = FunctionTransformer(np.log1p, validate=True)
    else:
        transformer_step = 'passthrough'

    # Build scaler step
    if scaler_method == 'standard':
        scaler_step = StandardScaler()
    elif scaler_method == 'minmax':
        scaler_step = MinMaxScaler()
    elif scaler_method == 'robust':
        scaler_step = RobustScaler()
    else:
        scaler_step = 'passthrough'

    # Only continuous cols get transformed/scaled; one-hot cols pass through untouched
    preprocessor = ColumnTransformer(transformers=[
        ('cont', Pipeline([
            ('transform', transformer_step),
            ('scale', scaler_step)
        ]), continuous_cols)
    ], remainder='passthrough')

    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        # LogisticRegression is scale-sensitive -> gives a REAL signal here,
        # unlike RandomForest which is invariant to this choice
        ('classifier', LogisticRegression(max_iter=2000, random_state=42))
    ])

    sample_x = x_train_final.sample(n=min(10000, len(x_train_final)), random_state=42)
    sample_y = y_train.loc[sample_x.index]


    try:
        # ... your existing code ...
        score = cross_val_score(pipeline, sample_x, sample_y, cv=3, scoring='roc_auc', n_jobs=-1).mean()
        return score
    except Exception as e:
        print(f"Trial failed: {e}")
        return float('-inf')  # tells Optuna this trial was bad, study continues
    
    
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=20)

print("=" * 60)
print(f"WINNING COMBINATION: {study.best_params}")
print(f"BEST CV ROC-AUC SCORE: {study.best_value:.4f}")
print("=" * 60)

[I 2026-08-02 23:22:03,766] A new study created in memory with name: no-name-f0c598c9-134d-4052-bbb7-31f2484ba233
[I 2026-08-02 23:22:08,248] Trial 0 finished with value: 0.5 and parameters: {'transform_method': 'yeo-johnson', 'scaler_method': 'none'}. Best is trial 0 with value: 0.5.
[I 2026-08-02 23:22:08,846] Trial 1 finished with value: 0.8336144411956518 and parameters: {'transform_method': 'none', 'scaler_method': 'standard'}. Best is trial 1 with value: 0.8336144411956518.
[I 2026-08-02 23:22:09,392] Trial 2 finished with value: 0.8485625894271477 and parameters: {'transform_method': 'log1p', 'scaler_method': 'robust'}. Best is trial 2 with value: 0.8485625894271477.
[I 2026-08-02 23:22:10,062] Trial 3 finished with value: 0.8496603785450705 and parameters: {'transform_method': 'log1p', 'scaler_method': 'standard'}. Best is trial 3 with value: 0.8496603785450705.
[I 2026-08-02 23:22:10,670] Trial 4 finished with value: 0.8485625894271477 and parameters: {'transform_method': 'log

WINNING COMBINATION: {'transform_method': 'log1p', 'scaler_method': 'standard'}
BEST CV ROC-AUC SCORE: 0.8497


In [27]:
from sklearn.preprocessing import StandardScaler


# 2. Create clean copies
x_train_scaled = x_train_final.copy()
x_test_scaled = x_test_final.copy()

# 3. Step A: Apply log1p safely BEFORE scaling (np.maximum(0, x) prevents -1 / NaNs)
for col in continuous_cols:
    x_train_scaled[col] = np.log1p(np.maximum(0, x_train_scaled[col]))
    x_test_scaled[col] = np.log1p(np.maximum(0, x_test_scaled[col]))

# 4. Step B: Apply StandardScaler AFTER log transformation
scaler = StandardScaler()

# Fit strictly on training set, transform both train & test
x_train_scaled[continuous_cols] = scaler.fit_transform(x_train_scaled[continuous_cols])
x_test_scaled[continuous_cols] = scaler.transform(x_test_scaled[continuous_cols])

print("✅ Power Transformation (log1p) and Standardization (StandardScaler) applied successfully!")
print(f"Features Transformed & Scaled : {len(continuous_cols)}")
print(f"x_train_scaled shape          : {x_train_scaled.shape}")
print(f"x_test_scaled shape           : {x_test_scaled.shape}")

✅ Power Transformation (log1p) and Standardization (StandardScaler) applied successfully!
Features Transformed & Scaled : 23
x_train_scaled shape          : (61060, 85)
x_test_scaled shape           : (26169, 85)


In [28]:
print("--- PREPROCESSING VERIFICATION ---")
print("Missing values in Train :", x_train_scaled.isnull().sum().sum())
print("Missing values in Test  :", x_test_scaled.isnull().sum().sum())
print("Infinite values in Train:", np.isinf(x_train_scaled[continuous_cols]).sum().sum())
print("Infinite values in Test :", np.isinf(x_test_scaled[continuous_cols]).sum().sum())
print("Values <= -1 check      :", (x_train_scaled[continuous_cols] <= -1).sum().sum(), "(Expected: Non-zero valid Z-scores, non-NaN)")

--- PREPROCESSING VERIFICATION ---
Missing values in Train : 0
Missing values in Test  : 0
Infinite values in Train: 0
Infinite values in Test : 0
Values <= -1 check      : 130064 (Expected: Non-zero valid Z-scores, non-NaN)


## Feature selection

In [29]:
from sklearn.feature_selection import (
    VarianceThreshold, chi2, SelectKBest, mutual_info_classif, RFE
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.preprocessing import MinMaxScaler

# ------------------------------------------------------------------------------
# 1. VARIANCE THRESHOLD (Filter: Dropping Near-Zero Variance)
# ------------------------------------------------------------------------------
var_thresh = VarianceThreshold(threshold=0.01)
var_thresh.fit(x_train_scaled)
var_selected_cols = x_train_scaled.columns[var_thresh.get_support()].tolist()

# ------------------------------------------------------------------------------
# 2. CHI-SQUARE TEST (Filter: Require non-negative features via MinMaxScaler)
# ------------------------------------------------------------------------------
# Chi2 requires strictly non-negative values
mm_scaler = MinMaxScaler()
x_train_non_neg = mm_scaler.fit_transform(x_train_scaled)

chi2_selector = SelectKBest(score_func=chi2, k=min(25, x_train_scaled.shape[1]))
chi2_selector.fit(x_train_non_neg, y_train)
chi2_selected_cols = x_train_scaled.columns[chi2_selector.get_support()].tolist()

# ------------------------------------------------------------------------------
# 3. MUTUAL INFORMATION (Filter: Non-linear statistical dependence)
# ------------------------------------------------------------------------------
# Sampled for speed on large notebook runs
sample_x = x_train_scaled.sample(n=min(10000, len(x_train_scaled)), random_state=42)
sample_y = y_train.loc[sample_x.index]

mi_scores = mutual_info_classif(sample_x, sample_y, random_state=42)
mi_series = pd.Series(mi_scores, index=x_train_scaled.columns).sort_values(ascending=False)
mi_selected_cols = mi_series.head(25).index.tolist()

# ------------------------------------------------------------------------------
# 4. RECURSIVE FEATURE ELIMINATION (Wrapper Method: RFE with Logistic Regression)
# ------------------------------------------------------------------------------
rfe_estimator = LogisticRegression(max_iter=1000, random_state=42)
rfe = RFE(estimator=rfe_estimator, n_features_to_select=25, step=5)
rfe.fit(sample_x, sample_y)
rfe_selected_cols = x_train_scaled.columns[rfe.get_support()].tolist()

print("✅ Feature Selection Techniques Executed Successfully!")

✅ Feature Selection Techniques Executed Successfully!


In [30]:
# Create consensus summary table
all_features = x_train_scaled.columns.tolist()
selection_summary = pd.DataFrame({'Feature': all_features})

selection_summary['Variance_Pass'] = selection_summary['Feature'].isin(var_selected_cols).astype(int)
selection_summary['Chi2_Top25'] = selection_summary['Feature'].isin(chi2_selected_cols).astype(int)
selection_summary['MutualInfo_Top25'] = selection_summary['Feature'].isin(mi_selected_cols).astype(int)
selection_summary['RFE_Top25'] = selection_summary['Feature'].isin(rfe_selected_cols).astype(int)

# Total score out of 4 methods
selection_summary['Consensus_Score'] = (
    selection_summary['Variance_Pass'] + 
    selection_summary['Chi2_Top25'] + 
    selection_summary['MutualInfo_Top25'] + 
    selection_summary['RFE_Top25']
)

selection_summary = selection_summary.sort_values(by='Consensus_Score', ascending=False).reset_index(drop=True)

print("🏆 TOP CONSENSUS FEATURES (Agreed upon by multiple techniques):")
print(selection_summary.head(20).to_string(index=False))

🏆 TOP CONSENSUS FEATURES (Agreed upon by multiple techniques):
                       Feature  Variance_Pass  Chi2_Top25  MutualInfo_Top25  RFE_Top25  Consensus_Score
                     lead_time              1           1                 1          1                4
   required_car_parking_spaces              1           1                 1          1                4
previous_bookings_not_canceled              1           1                 1          1                4
  market_segment_Offline TA/TO              1           1                 1          1                4
             room_type_changed              1           1                 1          1                4
                  country_freq              1           1                 1          1                4
      market_segment_Online TA              1           1                 1          1                4
       customer_type_Transient              1           1                 1          1                4
 

In [40]:
# Select features agreed upon by at least 2 methods (or set threshold to 1)
final_keep_cols = selection_summary[selection_summary['Consensus_Score'] >= 2]['Feature'].tolist()

# Fallback: if < 15 features meet score >= 2, keep top 25 overall
if len(final_keep_cols) < 15:
    final_keep_cols = selection_summary.head(25)['Feature'].tolist()

x_train_fs = x_train_scaled[final_keep_cols].copy()
x_test_fs = x_test_scaled[final_keep_cols].copy()

print("\n🎉 FEATURE SELECTION COMPLETE!")
print(f"Original Features : {x_train_scaled.shape[1]}")
print(f"Final Selected    : {x_train_fs.shape[1]}")
print(f"x_train_fs shape  : {x_train_fs.shape}")
print(f"x_test_fs shape   : {x_test_fs.shape}")
print(f"Columns selected  : {final_keep_cols}")


🎉 FEATURE SELECTION COMPLETE!
Original Features : 85
Final Selected    : 37
x_train_fs shape  : (61060, 37)
x_test_fs shape   : (26169, 37)
Columns selected  : ['lead_time', 'required_car_parking_spaces', 'previous_bookings_not_canceled', 'market_segment_Offline TA/TO', 'room_type_changed', 'country_freq', 'market_segment_Online TA', 'customer_type_Transient', 'deposit_type_Non Refund', 'total_of_special_requests', 'has_company', 'customer_type_Transient-Party', 'customer_type_Contract', 'adr_per_person', 'hotel_Resort Hotel', 'adr', 'distribution_channel_Direct', 'deposit_type_No Deposit', 'distribution_channel_TA/TO', 'previous_cancellations', 'booking_changes', 'market_segment_Corporate', 'meal_SC', 'distribution_channel_Corporate', 'market_segment_Direct', 'reserved_room_type_G', 'reserved_room_type_A', 'assigned_room_type_G', 'assigned_room_type_A', 'stays_in_week_nights', 'is_repeated_guest', 'agent', 'total_stay_nights', 'children', 'arrival_date_month_August', 'arrival_date_mo

In [41]:
# Run this in the feature engineering notebook, right where x_train_fs and y_train
# were last both in memory together (before saving to CSV)

print("x_train_fs index (first 10):", x_train_fs.index[:10].tolist())
print("y_train index (first 10):", y_train.index[:10].tolist())
print("Indexes match:", (x_train_fs.index == y_train.index).all())

# Also check the correlation THERE, before saving — to isolate whether
# the bug is in the feature engineering notebook or in the save/reload step
print("lead_time correlation (in feature eng notebook):", x_train_fs['lead_time'].corr(y_train))

x_train_fs index (first 10): [94051, 108249, 116820, 51992, 37145, 44749, 24730, 50392, 8425, 89261]
y_train index (first 10): [94051, 108249, 116820, 51992, 37145, 44749, 24730, 50392, 8425, 89261]
Indexes match: True
lead_time correlation (in feature eng notebook): 0.2372977012876671


## Pipeline in py script

In [1]:
# ============================================================
# CELL 1 — Setup
# ============================================================
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('..')

import pandas as pd
from sklearn.model_selection import train_test_split
from src.hotel_booking_cancelation_prediction.feature_pipeline import (
    run_feature_pipeline,
    save_processed_data
)

In [3]:
# ============================================================
# CELL 2 — Raw data → cleaning → split
# ============================================================
from src.hotel_booking_cancelation_prediction.cleaning import pandas_cleaning
hotel_data = pd.read_csv('../data/raw/hotel_bookings.csv')  # adjust path to yours

df = pandas_cleaning(hotel_data)

if 'company' in df.columns:
    df['has_company'] = df['company'].notnull().astype('int64')
    df = df.drop(columns=['company'])

x = df.drop(columns=['is_canceled'])
y = df['is_canceled']

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.3, random_state=0, stratify=y
)

print("x_train:", x_train.shape)
print("x_test :", x_test.shape)

x_train: (61060, 30)
x_test : (26169, 30)


In [4]:
# ============================================================
# CELL 3 — Run the full pipeline (no feature selection — full 85 cols)
# ============================================================
x_train_scaled, x_test_scaled = run_feature_pipeline(x_train, x_test)

print("x_train_scaled:", x_train_scaled.shape)
print("x_test_scaled :", x_test_scaled.shape)

x_train_scaled: (61060, 85)
x_test_scaled : (26169, 85)


In [5]:
from src.hotel_booking_cancelation_prediction.feature_pipeline import SELECTED_FEATURES
print(len(SELECTED_FEATURES))

37


In [6]:
missing = [c for c in SELECTED_FEATURES if c not in x_train_scaled.columns]
print("Missing:", missing)
print("Total x_train_scaled columns:", x_train_scaled.shape[1])

Missing: []
Total x_train_scaled columns: 85


In [7]:
# ============================================================
# CELL 4 — Feature selection (proper function, index-safe)
# ============================================================
from src.hotel_booking_cancelation_prediction.feature_pipeline import select_features

x_train_fs, x_test_fs = select_features(x_train_scaled, x_test_scaled)

print("x_train_fs:", x_train_fs.shape)
print("x_test_fs :", x_test_fs.shape)


x_train_fs: (61060, 37)
x_test_fs : (26169, 37)


In [10]:
# ============================================================
# CELL 5 — Verify + save
# ============================================================
if not (x_train_scaled.index == y_train.index).all():
    raise ValueError("x_train_scaled and y_train indices don't match!")

print("lead_time correlation:", x_train_scaled['lead_time'].corr(y_train))
# Should print ~0.237

save_processed_data(x_train_fs, x_test_fs, y_train, y_test)

lead_time correlation: 0.2372977012876671
Saved ../data/processed\train_processed.csv — shape (61060, 38)
Saved ../data/processed\test_processed.csv — shape (26169, 38)


('../data/processed\\train_processed.csv',
 '../data/processed\\test_processed.csv')